In [ ]:
import requests
from requests import Response
from pandas import DataFrame
import pandas as pd 
from pathlib import Path
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
from reportlab.platypus import PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle

def export_to_pdf(dataset: pd.DataFrame, output_path: Path) -> None:
    # 註冊可顯示中文的字型（macOS / Linux 常可直接使用）
    font_name = "STSong-Light"
    pdfmetrics.registerFont(UnicodeCIDFont(font_name))

    # 挑選適合放進報表的欄位，避免欄位太多導致版面混亂
    preferred_columns = ["sno", "sna", "sarea", "ar", "tot", "sbi", "bemp", "mday"]
    columns = [col for col in preferred_columns if col in df.columns]

    if not columns:
        print("找不到可輸出的欄位，無法建立 PDF。")
        return

    table_rows = df[columns].fillna("").astype(str).values.tolist()
    rows_per_page = 35

    doc = SimpleDocTemplate(
        str(output_path),
        pagesize=landscape(A4),
        rightMargin=18,
        leftMargin=18,
        topMargin=18,
        bottomMargin=18,
    )

    styles = getSampleStyleSheet()
    title_style = styles["Title"]
    title_style.fontName = font_name

    story = [
        Paragraph("YouBike 即時資料報表", title_style),
        Spacer(1, 12),
    ]

    for index in range(0, len(table_rows), rows_per_page):
        chunk = table_rows[index:index + rows_per_page]
        table_data = [columns] + chunk

        col_widths = []
        for col in columns:
            if col in ("sna", "ar"):
                col_widths.append(135)
            elif col == "mday":
                col_widths.append(120)
            else:
                col_widths.append(62)

        table = Table(table_data, repeatRows=1, colWidths=col_widths)
        table.setStyle(
            TableStyle(
                [
                    ("FONTNAME", (0, 0), (-1, -1), font_name),
                    ("FONTSIZE", (0, 0), (-1, -1), 8),
                    ("BACKGROUND", (0, 0), (-1, 0), colors.lightblue),
                    ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
                    ("GRID", (0, 0), (-1, -1), 0.3, colors.grey),
                    ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                ]
            )
        )

        story.append(table)

        if index + rows_per_page < len(table_rows):
            story.append(PageBreak())

    doc.build(story)
    print(f"PDF 已產生：{output_path}")

# 台北市 YouBike 2.0 的 Web API 網址
url:str = "https://tcgbusfs.blob.core.windows.net/dotapp/youbike/v2/youbike_immediate.json"

# 使用 requests 套件裡面的 get 函式，執行後會傳出 Response 的實體
response:Response = requests.get(url) 

if response.status_code == 200: # 使用 Response 裡的 Property 叫 status_code，如果取得的數字是 200 代表下載成功，如果不是則代表下載失敗
    data:list[dict] = response.json() # 使用 Response 實體的 json() 方法，會傳出 list 的資料結構

    # list[dict] -> DataFrame
    df:DataFrame = pd.DataFrame(data=data)

    print(df.tail())

    #output_file = Path(__file__).with_name("youbike_report_ipynb.pdf") ##ipynb裡name '__file__' is not defined 要換一種寫法
    outputfile = Path.cwd().with_name("youbike_report_ipynb.pdf") #設定輸出的絕對路徑
    export_to_pdf(df, outputfile)

df.iloc[0] #data的第一筆資料

            sno                      sna   sarea                 mday  \
1759  500119094     YouBike2.0_臺大大一女餐廳廣場  臺大公館校區  2026-06-13 11:21:03   
1760  500119095         YouBike2.0_臺大學新館  臺大公館校區  2026-06-13 11:23:02   
1761  500119096      YouBike2.0_臺大水源舍區C棟  臺大公館校區  2026-06-13 10:56:02   
1762  500119097         YouBike2.0_臺大人文館  臺大公館校區  2026-06-13 11:19:03   
1763  500119098  YouBike2.0_臺大數學研究中心(東側)  臺大公館校區  2026-06-13 11:23:02   

               ar   sareaen  \
1759  臺大大一女餐廳廣場後方  NTU Dist   
1760       臺大學新館旁  NTU Dist   
1761   臺大水源舍區C棟西側  NTU Dist   
1762     臺大人文館東北側  NTU Dist   
1763   臺大數學研究中心東側  NTU Dist   

                                                  snaen  \
1759    YouBike2.0_NTU Freshman Women's Dorm Restaurant   
1760                  YouBike2.0_NTU MK Innovation Hall   
1761                   YouBike2.0_NTU Shui-Yuan Dorms C   
1762         YouBike2.0_NTU The College of Liberal Arts   
1763  YouBike2.0_Mathematics National Taiwan Univers...   

                   

sno                                                  500101001
sna                                         YouBike2.0_捷運科技大樓站
sarea                                                      大安區
mday                                       2026-06-13 11:25:03
ar                                                 復興南路二段235號前
sareaen                                             Daan Dist.
snaen                     YouBike2.0_MRT Technology Bldg. Sta.
aren                             No.235， Sec. 2， Fuxing S. Rd.
act                                                          1
srcUpdateTime                              2026-06-13 11:25:52
updateTime                                 2026-06-13 11:25:52
infoTime                                   2026-06-13 11:25:03
infoDate                                            2026-06-13
Quantity                                                    28
available_rent_bikes                                        18
latitude                                              2

In [3]:
print(Path.cwd())

c:\Users\user\Documents\GitHub\procject_2026\13JUN
